# PointNet++ Training experimental

This notebook serves the purpose of creating the training routine for the PointNet++ to encode point clouds into the same latent space as the CAD sequence encoding done by the DeepCAD model.

The pointnet venv is located at "Point-Cloud-Reconstruction/pointnet.pytorch", but the model used is in "Point-Cloud-Reconstruction/Pointnet_Pointnet2_pytorch"

## 1. Check point cloud data
We will use open3d to inspect the point cloud files. During the conversion of the CAD sequences to point clouds errors appeared, which is why I assume there must be some corrupt point cloud files, which need to be excluded.

- Load train/val/test split
- create train/val/test pc path lists

In [130]:
import open3d as o3d
import os
from tqdm import tqdm

In [131]:
DATA_ROOT = "../data/pc_cad"
SPLIT = "../data/train_val_test_split.json"

In [132]:
with open(SPLIT, "r") as fp:
    all_data = json.load(fp)
print(f"Number of samples that should be in the split: {len(train)+len(val)+len(test)}")
for phase in all_data.keys():
    print(phase, len(all_data[phase]))

Number of samples that should be in the split: 178238
train 161240
validation 8946
test 8052


In [133]:
train = [os.path.join(DATA_ROOT, f"{idx}.ply") for idx in all_data['train']]
val = [os.path.join(DATA_ROOT, f"{idx}.ply") for idx in all_data['validation']]
test = [os.path.join(DATA_ROOT, f"{idx}.ply") for idx in all_data['test']]

Now we need to check if the point cloud files in the split are actually there and not corrupt.

In [134]:
from glob import glob
directory = DATA_ROOT
file_pattern = "**/*.ply"
all_files = glob(f"{directory}/{file_pattern}", recursive=True)
print(f"Total number of files: {len(all_files)}")
print(f"Missing/corrupt files: {len(train)+len(val)+len(test)-len(all_files)}")

Total number of files: 177948
Missing/corrupt files: 290


Apparently the conversion from json to point cloud did not work in 290 cases. These have to be removed from the split.

In [144]:
corrupt_files = []
correct_files = []
for pc_file in train:
    try:
        pc = o3d.io.read_point_cloud(pc_file)
        if not pc.has_points():
            corrupt_files.append(pc_file)
    except Exception as e:
        corrupt_files.append(pc_file)
    if not os.path.exists(pc_file):
        corrupt_files.append(pc_file)
    else:
        correct_files.append(pc_file)

RPly: Unable to open file
RPly: Unable to open file
RPly: Unable to open file


[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0086/00866760.ply
[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0022/00222961.ply
[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0083/00837927.ply


RPly: Unable to open file
RPly: Unable to open file


[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0070/00704820.ply
[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0087/00870042.ply


RPly: Unable to open file


...

[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0074/00743986.ply
[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0064/00649141.ply


RPly: Unable to open file
RPly: Unable to open file
RPly: Unable to open file


[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0043/00438205.ply
[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0044/00445989.ply
[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0028/00284028.ply


RPly: Unable to open file


[Open3D WARNING] Read PLY failed: unable to open file: ../data/pc_cad/0046/00463296.ply


In [145]:
len(corrupt_files), len(correct_files)

(516, 160982)

Corrupt files in (test: 28, val: 36, train: 516) -> Total = 290 anscheinend mehr?? Ja anscheinend gab es 290 Fälle in denen nicht konvertiert werden konnte, aber noch ein paar mehr Fälle wo zwar konvertiert wurde, aber die point cloud an sich halt corrupt ist.

NEXT TIME: Herausfinden, welche PC corrupt ist und welche nicht konvertiert werden konnten (290), damit sollten (516-290) = 226 corrupt sein. Dann eine fertige liste mit funktionierenden point clouds erstellen. Am ende zum beispiel ein script "prepare pointnet++ training data.py" oder sowas erstellen.

In [107]:
print(no_points)

['../data/pc_cad/0056/00566337.ply']


Test: 14 pc's waren entweder point cloud conversion fail oder create cad fail, alle anderen korrekt
Val: 18 pc's waren entweder point cloud conversion fail oder create cad fail, alle anderen korrekt
Train: Irgendwo findet ein Parallels problem statt (0, 78950, 1, 82290)Es ist die file 0011/00116212

Error kommt von process_one -> create_CAD (in visualize.py)

Laut stackoverflow soll man mit ulimit -s stack anschauen und mit ulimit -s \<neuerWert> erhöhen

Chat GPT: "On macOS (and other Unix-like operating systems), the ulimit -s command is used to query or set the stack size limit for processes. The stack size determines the amount of memory allocated for a program's stack, which is used for function calls, local variables, and control flow."



In [113]:
problematic_file = "/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/cad_json/0011/00116212.json"
try:
    with open(problematic_file, "r") as f:
        data = json.load(f)
    print("File loaded successfully.")
    print(json.dumps(data, indent=4))  # Pretty-print the JSON content.
except json.JSONDecodeError as e:
    print(f"JSON decoding error: {e}")
except Exception as e:
    print(f"Error reading file: {e}")

File loaded successfully.
{
    "entities": {
        "FRAcwEqNExbvyOz_2": {
            "name": "Extrude 3",
            "type": "ExtrudeFeature",
            "profiles": [
                {
                    "profile": "JNC",
                    "sketch": "FRSAyqbsz52iKa9_2"
                }
            ],
            "extent_two": {
                "distance": {
                    "type": "ModelParameter",
                    "role": "AgainstDistance",
                    "name": "none",
                    "value": 0.0
                },
                "type": "DistanceExtentDefinition",
                "taper_angle": {
                    "type": "ModelParameter",
                    "role": "Side2TaperAngle",
                    "name": "none",
                    "value": 0.0
                }
            },
            "extent_one": {
                "distance": {
                    "type": "ModelParameter",
                    "role": "AlongDistance",
                   

In [ ]:
from OCC.Core.BRepCheck import BRepCheck_Analyzer

def is_valid_shape(shape):
    analyzer = BRepCheck_Analyzer(shape)
    return analyzer.IsValid()
